Adjust import path:


In [ ]:
import sys
from pathlib import Path

repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "ptps_wildfire_demo").is_dir()
)
sys.path.insert(0, str(repo_root))

In [ ]:
import httpx

from ptps_wildfire_demo.proxy.resolver import Resolver

client = httpx.AsyncClient()
resolver = Resolver(client)

In [ ]:
sample = resolver.drp_rescues.sample(100)
sample

,dataset,dataset_id,status,url,source_website,organization,agency,download_date,size,maintainer,download_location,file_type,notes,metadata_available,metadata_url
734,National Survey of Organ Donation Attitudes an...,876,Finished,https://data.hrsa.gov/topics/health-systems/or...,data.hrsa.gov,Health Resources and Services Administration (...,Department of Health and Human Services,2025-04-18,2.36,"DRP, DL",https://www.datalumos.org/datalumos/project/22...,ZIP,<NA>,<NA>,<NA>
4418,Historical tree surveys of the Umatilla Nation...,6599,Finished,https://www.fs.usda.gov/rds/archive/catalog/RD...,fs.usda.gov,US Forest Service,U.S. Department of Agriculture,2026-06-05,0.0059,"DRP, DL",https://www.datalumos.org/datalumos/project/24...,"PDF, ZIP",<NA>,yes,<NA>
211,Paleoclimatology: Paleoceanography,405,Finished,https://www.ncei.noaa.gov/products/paleoclimat...,ncei.noaa.gov,National Oceanic and Atmospheric Administration,Department of Commerce,<NA>,69.0,EDGI,<NA>,"XLSX, PDF, JSON",Daro/PEDP S3 Bucket,<NA>,<NA>
91,Social Wellbeing Report,102,Finished,https://www.imls.gov/research-tools/data-colle...,imls.gov,Institute of Museum and Library Services,Institute of Museum and Library Services,2025-02-11,0.0,"DRP, DL",https://www.datalumos.org/datalumos/project/21...,ZIP,"report + case studies, in combined folder",<NA>,<NA>
908,State Summaries_Iowa,1064,Finished,https://www.data.va.gov/stories/s/7g2p-v3sr,data.va.gov,Office of Information and Technology - IT Oper...,Department of Veterans Affairs,2025-04-24,0.0,"DRP, DL",https://www.datalumos.org/datalumos/project/22...,"CSV, PDF",<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2813,PONE-D-15-23803,4796,Finished,https://data.cdc.gov/dataset/PONE-D-15-23803/k...,data.cdc.gov,Centers for Disease Control and Prevention (CDC),Department of Health and Human Services,2026-01-13,0.0021,"DRP, DL",https://www.datalumos.org/datalumos/project/24...,"PDF, CSV",<NA>,<NA>,<NA>
1496,Inhalation of polycarbonate emissions generate...,1701,Finished,https://data.cdc.gov/National-Institute-for-Oc...,data.cdc.gov,Centers for Disease Control and Prevention (CDC),Department of Health and Human Services,<NA>,<NA>,DL,https://www.datalumos.org/datalumos/project/23...,<NA>,<NA>,<NA>,<NA>
288,National Survey of College Graduates - 2021 Da...,334,Finished,https://www.census.gov/,census.gov,Census Bureau,Department of Commerce,2025-01-31,0.0,ICPSR,https://www.dropbox.com/scl/fo/b4525g7cf1caxsp...,"PDF, XLSX",<NA>,<NA>,<NA>
1318,NCVAS State Summary Pennsylvania FY2021,1470,Finished,https://www.data.va.gov/stories/s/p8ge-7syx,data.va.gov,Office of Information and Technology - IT Oper...,Department of Veterans Affairs,2025-08-31,0.0012,"DRP, DL",https://www.datalumos.org/datalumos/project/23...,"CSV, PDF",<NA>,<NA>,<NA>


In [ ]:
import asyncio
import re

import httpx
import pandas as pd


async def get_status(url: str) -> int | str:
    # some URLs are missing the protocol
    if not re.search(r"^https?://", url):
        url = f"https://{url}"

    try:
        response = await client.head(url, follow_redirects=True, timeout=20)
        return response.status_code
    except httpx.TimeoutException:
        return "timeout"
    except httpx.HTTPError as e:
        return f"error: {e}"


async def get_statuses() -> list[int | str]:
    return await asyncio.gather(*(get_status(url) for url in sample["url"]))


statuses = await get_statuses()
sample["status"] = statuses


pd.set_option("display.max_colwidth", 200)
sample[["url", "status"]]


,url,status
734,https://data.hrsa.gov/topics/health-systems/organ_donation_opinion_survey-data,200
4418,https://www.fs.usda.gov/rds/archive/catalog/RDS-2020-0051,timeout
211,https://www.ncei.noaa.gov/products/paleoclimatology/paleoceanography,error: Server disconnected without sending a response.
91,https://www.imls.gov/research-tools/data-collection/social-wellbeing-report,timeout
908,https://www.data.va.gov/stories/s/7g2p-v3sr,200
...,...,...
2813,https://data.cdc.gov/dataset/PONE-D-15-23803/krkm-t59m/about_data,200
1496,https://data.cdc.gov/National-Institute-for-Occupational-Safety-and-Hea/Inhalation-of-polycarbonate-emissions-generated-du/dtz3-sij3,200
288,https://www.census.gov/,200
1318,https://www.data.va.gov/stories/s/p8ge-7syx,200


In [ ]:
sample[sample["status"] != 200][["url", "status"]]

,url,status
4418,https://www.fs.usda.gov/rds/archive/catalog/RDS-2020-0051,timeout
211,https://www.ncei.noaa.gov/products/paleoclimatology/paleoceanography,error: Server disconnected without sending a response.
91,https://www.imls.gov/research-tools/data-collection/social-wellbeing-report,timeout
735,https://mchb.hrsa.gov/data-research/chartbooks,403
4062,https://www.fs.usda.gov/rds/archive/catalog/RDS-2024-0078,timeout
1027,https://www.nhtsa.gov/nhtsa-datasets-and-apis,403
3755,https://agdatacommons.nal.usda.gov/articles/dataset/Data_from_Carbon_Fluxes_from_a_Spring_Wheat-Corn-Soybean_Crop_Rotation_Under_No-Tillage_Management/24856911,202
1965,hifld-geoplatform.hub.arcgis.com,404
112,https://www.huduser.gov/portal/datasets/fmr.html,202
1135,https://www.climate.gov/maps-data/data-snapshots/data-source/temperature-global-yearly-difference-average,202
